# 0. Basis (Q, K, V Matrisleri)

Nedir? Nasıl Hesaplanır? Nasıl Kullanılır?

In [1]:
import numpy as np

# --- Fonksiyonlar ---
def softmax(x):
    """Numerik olarak stabil bir softmax hesaplaması."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def print_attention(tokens, attention_matrix):
    """Dikkat matrisini okunaklı bir şekilde yazdırır."""
    header = "         " + "  ".join([f"{token:<8}" for token in tokens])
    print(header)
    print("-" * len(header))
    for i, token in enumerate(tokens):
        row_str = f"{token:<9}"
        for val in attention_matrix[i]:
            row_str += f"{val:.4f}  "
        print(row_str)
    print("\n")

# --- Adım 0: Başlangıç Ayarları ---
print("--- BAŞLANGIÇ: RASTGELE MATRİSLER ---")
# Model ve vektör boyutları
d_model = 4
d_k = 3
np.random.seed(42) # Tekrarlanabilirlik için

# BAŞLANGIÇTAKİ RASTGELE AĞIRLIK MATRİSLERİ
# Bu matrisler tüm eğitim süreci boyunca güncellenecek.
W_q = np.random.rand(d_model, d_k)
W_k = np.random.rand(d_model, d_k)
W_v = np.random.rand(d_model, d_k) # Bu simülasyonda W_v'yi sabit tutacağız

print("Başlangıç W_q Matrisi:\n", np.round(W_q, 2))
print("\nBaşlangıç W_k Matrisi:\n", np.round(W_k, 2), "\n")
print("="*50)

# --- Adım 1: Eğitim Simülasyonu ---
# Basit bir eğitim veri seti ve hedefler
egitim_verisi = [
    {"metin": "akıllı köpek topu getirdi", "kaynak_idx": 1, "hedef_idx": 3}, # köpek -> getirdi
    {"metin": "yorgun kedi minderde uyudu", "kaynak_idx": 1, "hedef_idx": 3}, # kedi -> uyudu
]

iterasyon_sayisi = 3
ogrenme_orani = 0.1 # Matrisleri ne kadar değiştireceğimizi belirler

for i in range(iterasyon_sayisi):
    print(f"\n########### ITERASYON {i+1} ###########\n")
    
    for veri in egitim_verisi:
        metin = veri["metin"]
        kaynak_idx = veri["kaynak_idx"]
        hedef_idx = veri["hedef_idx"]
        tokenler = metin.split()
        
        print(f"--- Girdi Metni: '{metin}' ---")
        print(f"Hedef: '{tokenler[kaynak_idx]}' kelimesi '{tokenler[hedef_idx]}' kelimesine dikkat etsin.\n")
        
        # 1. Her girdi için rastgele kelime vektörleri (embedding) oluştur
        X = np.random.rand(len(tokenler), d_model)
        
        # 2. İleri Yayılım (Forward Pass) - Mevcut matrislerle dikkat skorlarını hesapla
        Q = np.dot(X, W_q)
        K = np.dot(X, W_k)
        
        skorlar = np.dot(Q, K.T) / np.sqrt(d_k)
        dikkat_agirliklari = softmax(skorlar)
        
        print(f"--- GÜNCELLEME ÖNCESİ DİKKAT MATRİSİ (İterasyon {i+1}) ---")
        print_attention(tokenler, dikkat_agirliklari)
        
        # 3. Hata Hesaplama (Simülasyon)
        # Hedef dikkat skorunun 1.0 olmasını istiyoruz. Mevcut skor ne kadar düşükse, hata o kadar büyük.
        mevcut_skor = dikkat_agirliklari[kaynak_idx, hedef_idx]
        hata = 1.0 - mevcut_skor
        print(f"'{tokenler[kaynak_idx]}' -> '{tokenler[hedef_idx]}' için mevcut dikkat skoru: {mevcut_skor:.4f}")
        print(f"Hesaplanan Hata (1.0 - skor): {hata:.4f}\n")
        
        # 4. Ağırlıkları Güncelleme (Simülasyon)
        # Bu, gerçek backpropagation'ın ÇOK basitleştirilmiş bir taklididir.
        # Fikir: Hedef skoru artırmak için W_q ve W_k'yi küçük bir miktar "dürtelim".
        # Düzeltme miktarını hata ve öğrenme oranı ile ölçekliyoruz.
        duzeltme_miktari = hata * ogrenme_orani
        
        # Sorumlu olan kelimelerin giriş vektörlerini (X) kullanarak güncelleme yapıyoruz.
        # Bu, güncellemenin doğru yönde olmasına yardımcı olur.
        W_q_guncellemesi = np.outer(X[kaynak_idx], Q[kaynak_idx]) * duzeltme_miktari
        W_k_guncellemesi = np.outer(X[hedef_idx], K[hedef_idx]) * duzeltme_miktari
        
        W_q += W_q_guncellemesi
        W_k += W_k_guncellemesi
        
        print(">>> Ağırlık Matrisleri Güncellendi! <<<\n")
        
        # 5. Doğrulama: Güncellenmiş matrislerle skorlar nasıl değişti?
        print(f"--- GÜNCELLEME SONRASI DOĞRULAMA (İterasyon {i+1}) ---")
        Q_yeni = np.dot(X, W_q)
        K_yeni = np.dot(X, W_k)
        skorlar_yeni = np.dot(Q_yeni, K_yeni.T) / np.sqrt(d_k)
        dikkat_yeni = softmax(skorlar_yeni)
        
        yeni_skor = dikkat_yeni[kaynak_idx, hedef_idx]
        print(f"'{tokenler[kaynak_idx]}' -> '{tokenler[hedef_idx]}' için YENİ dikkat skoru: {yeni_skor:.4f}")
        print(f"Skor Artışı: {(yeni_skor - mevcut_skor):.4f}\n")
        print_attention(tokenler, dikkat_yeni)
        print("-" * 50)

print("\n\n--- EĞİTİM SONRASI FİNAL MATRİSLER ---")
print("Rastgele başlayan matrisler, hedefler doğrultusunda güncellenerek evrimleşti.")
print("Final W_q Matrisi:\n", np.round(W_q, 2))
print("\nFinal W_k Matrisi:\n", np.round(W_k, 2))

--- BAŞLANGIÇ: RASTGELE MATRİSLER ---
Başlangıç W_q Matrisi:
 [[0.37 0.95 0.73]
 [0.6  0.16 0.16]
 [0.06 0.87 0.6 ]
 [0.71 0.02 0.97]]

Başlangıç W_k Matrisi:
 [[0.83 0.21 0.18]
 [0.18 0.3  0.52]
 [0.43 0.29 0.61]
 [0.14 0.29 0.37]] 


########### ITERASYON 1 ###########

--- Girdi Metni: 'akıllı köpek topu getirdi' ---
Hedef: 'köpek' kelimesi 'getirdi' kelimesine dikkat etsin.

--- GÜNCELLEME ÖNCESİ DİKKAT MATRİSİ (İterasyon 1) ---
         akıllı    köpek     topu      getirdi 
-----------------------------------------------
akıllı   0.2169  0.1953  0.2327  0.3552  
köpek    0.2230  0.1836  0.2271  0.3663  
topu     0.2198  0.1859  0.2278  0.3664  
getirdi  0.1925  0.1602  0.2146  0.4327  


'köpek' -> 'getirdi' için mevcut dikkat skoru: 0.3663
Hesaplanan Hata (1.0 - skor): 0.6337

>>> Ağırlık Matrisleri Güncellendi! <<<

--- GÜNCELLEME SONRASI DOĞRULAMA (İterasyon 1) ---
'köpek' -> 'getirdi' için YENİ dikkat skoru: 0.3933
Skor Artışı: 0.0269

         akıllı    köpek     topu      g

## 0.0.1 Basis : Yapay Sinir Ağı Kullanımı
    

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# --- Adım 0: Veri ve Kurulum ---
print("--- Adım 0: Veri ve Kurulum ---")
# Cümlemiz ve hedefimiz
sentence = "akıllı köpek topu getirdi"
tokens = sentence.split()
source_token = 'köpek'
target_token = 'getirdi'

print(f"Cümle: '{sentence}'")
print(f"Hedef: '{source_token}' kelimesi -> '{target_token}' kelimesine dikkat etsin.\n")

# Kelime dağarcığı oluşturma
vocab = sorted(list(set(tokens)))
word_to_ix = {word: i for i, word in enumerate(vocab)}
vocab_size = len(vocab)

# Girdi ve hedefi PyTorch tensörlerine çevirme
input_indices = torch.LongTensor([word_to_ix[w] for w in tokens])
source_idx = tokens.index(source_token)
target_idx = tokens.index(target_token)

# Hedef dikkat matrisi: (kaynak, hedef) pozisyonunda 1.0, diğer yerlerde 0.0
target_attention = torch.zeros(len(tokens), len(tokens))
target_attention[source_idx, target_idx] = 1.0

print("Veri PyTorch tensörlerine çevrildi.")
print(f"Girdi İndeksleri: {input_indices}")
print(f"Hedef Dikkat Matrisi (istenilen durum):\n{target_attention}\n")
print("="*60)


# --- Adım 1: Basit Sinir Ağı Modelini Tanımlama ---
print("\n--- Adım 1: PyTorch ile Basit Dikkat Ağı Modeli ---")

class SimpleAttentionNetwork(nn.Module):
    def __init__(self, vocab_size, d_model, d_k):
        super(SimpleAttentionNetwork, self).__init__()
        self.d_k = d_k
        
        # Kelime embedding'leri için katman (Bu W1 matrisine karşılık gelir)
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Q, K, V dönüşümleri için Doğrusal katmanlar (Bunlar W_q, W_k, W_v matrisleridir)
        self.q_transform = nn.Linear(d_model, d_k, bias=False)
        self.k_transform = nn.Linear(d_model, d_k, bias=False)
        # Not: Bu basit örnekte V'yi kullanmayacağız ama tam bir modelde olurdu.

    def forward(self, input_indices):
        # 1. Girdi indekslerinden embedding vektörlerini al
        embedded_vectors = self.embedding(input_indices)  # Shape: (seq_len, d_model)
        
        # 2. Embedding'leri Q ve K uzaylarına yansıt
        Q = self.q_transform(embedded_vectors)  # Shape: (seq_len, d_k)
        K = self.k_transform(embedded_vectors)  # Shape: (seq_len, d_k)
        
        # 3. Dikkat skorlarını hesapla
        # (seq_len, d_k) @ (d_k, seq_len) -> (seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        return attention_scores, Q, K

# Model hiperparametreleri
D_MODEL = 16
D_K = 8
LEARNING_RATE = 0.1
EPOCHS = 100

# Modeli, kayıp fonksiyonunu ve optimize ediciyi oluştur
model = SimpleAttentionNetwork(vocab_size, D_MODEL, D_K)
criterion = nn.MSELoss() # Ortalama Karesel Hata: Modelin çıktısı ile hedef arasındaki farkı ölçer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Sinir ağı modeli, kayıp fonksiyonu ve optimize edici hazır.\n")
print("="*60)

# --- Adım 2: Eğitim Döngüsü ---
print("\n--- Adım 2: Sinir Ağı Eğitim Döngüsü ---")

def print_attention_torch(tokens, attention_tensor):
    """PyTorch tensörünü alıp dikkat matrisini yazdırır."""
    attention_np = attention_tensor.detach().numpy() # Gradyan takibini bırakıp numpy'a çevir
    # Softmax uygulayarak olasılıklara çevir
    e_x = np.exp(attention_np - np.max(attention_np, axis=-1, keepdims=True))
    probs = e_x / np.sum(e_x, axis=-1, keepdims=True)
    
    header = "         " + "  ".join([f"{token:<8}" for token in tokens])
    print(header)
    print("-" * len(header))
    for i, token in enumerate(tokens):
        row_str = f"{token:<9}"
        for val in probs[i]:
            row_str += f"{val:.4f}  "
        print(row_str)
    print("\n")


# Eğitimin başlangıcındaki duruma bakalım
print("--- EĞİTİM ÖNCESİ DURUM ---")
initial_scores, _, _ = model(input_indices)
print("Başlangıçtaki Dikkat Matrisi (Rastgele):")
print_attention_torch(tokens, initial_scores)
print(f"Başlangıçtaki W_q matrisi:\n{np.round(model.q_transform.weight.data.numpy(), 2)}\n")


for epoch in range(EPOCHS):
    # İleri Yayılım: Modelin mevcut ağırlıklarla tahmin yapması
    output_scores, _, _ = model(input_indices)
    
    # Kayıp Hesabı: Modelin tahmini ile bizim hedefimiz arasındaki fark
    loss = criterion(output_scores, target_attention)
    
    # Geriye Yayılım ve Optimizasyon
    optimizer.zero_grad()  # Önceki adımların gradyanlarını temizle
    loss.backward()        # Kaybı azaltmak için gradyanları hesapla (AutoGrad)
    optimizer.step()       # Optimize edici ile ağırlıkları (W_q, W_k, embedding) güncelle
    
    if (epoch + 1) % 25 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Kayıp (Loss): {loss.item():.4f}")

print("\n--- EĞİTİM SONRASI DURUM ---")
final_scores, final_Q, final_K = model(input_indices)
print("\nEğitim Sonrası Nihai Dikkat Matrisi:")
print_attention_torch(tokens, final_scores)

print("Q ve K matrisleri artık hedefe yönelik dönüşümler yapmayı 'öğrendi'.")
print(f"\nEğitim Sonrası Final W_q matrisi:\n{np.round(model.q_transform.weight.data.numpy(), 2)}")
print(f"\nEğitim Sonrası Final W_k matrisi:\n{np.round(model.k_transform.weight.data.numpy(), 2)}")

--- Adım 0: Veri ve Kurulum ---
Cümle: 'akıllı köpek topu getirdi'
Hedef: 'köpek' kelimesi -> 'getirdi' kelimesine dikkat etsin.

Veri PyTorch tensörlerine çevrildi.
Girdi İndeksleri: tensor([0, 2, 3, 1])
Hedef Dikkat Matrisi (istenilen durum):
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


--- Adım 1: PyTorch ile Basit Dikkat Ağı Modeli ---
Sinir ağı modeli, kayıp fonksiyonu ve optimize edici hazır.


--- Adım 2: Sinir Ağı Eğitim Döngüsü ---
--- EĞİTİM ÖNCESİ DURUM ---
Başlangıçtaki Dikkat Matrisi (Rastgele):
         akıllı    köpek     topu      getirdi 
-----------------------------------------------
akıllı   0.1420  0.3204  0.0951  0.4425  
köpek    0.2354  0.2443  0.3029  0.2174  
topu     0.2099  0.2022  0.1572  0.4306  
getirdi  0.2530  0.2469  0.2639  0.2361  


Başlangıçtaki W_q matrisi:
[[-0.09  0.21 -0.05  0.24 -0.23 -0.11  0.04 -0.14 -0.01 -0.24 -0.01  0.23
  -0.14  0.    0.01 -0.19]
 [-0.17  0.11 -0.14  0.13 -0.1

## 0.1. Basis K ve V ilişkisi

In [3]:
import numpy as np

# --- Fonksiyonlar ---
def softmax(x):
    """Numerik olarak stabil bir softmax hesaplaması."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def print_attention(tokens, attention_matrix):
    """Dikkat matrisini okunaklı bir şekilde yazdırır."""
    header = "         " + "  ".join([f"{token:<8}" for token in tokens])
    print(header)
    print("-" * len(header))
    for i, token in enumerate(tokenler):
        row_str = f"{token:<9}"
        for val in attention_matrix[i]:
            row_str += f"{val:.4f}  "
        print(row_str)
    print("\n")

# --- Adım 0: Başlangıç Ayarları ---
print("--- BAŞLANGIÇ: RASTGELE MATRİSLER ---")
d_model = 4
d_k = 3 # K, Q ve V'nin boyutunu aynı tutuyoruz (d_k = d_v)
np.random.seed(42)

W_q = np.random.rand(d_model, d_k)
W_k = np.random.rand(d_model, d_k)
W_v = np.random.rand(d_model, d_k)

print("Başlangıç W_q Matrisi:\n", np.round(W_q, 2))
print("\nBaşlangıç W_k Matrisi:\n", np.round(W_k, 2))
print("\nBaşlangıç W_v Matrisi:\n", np.round(W_v, 2), "\n")
print("="*60)

# --- Adım 1: Eğitim Simülasyonu ---
egitim_verisi = [
    {"metin": "akıllı köpek topu getirdi", "kaynak_idx": 1, "hedef_idx": 3}, # köpek -> getirdi
]

iterasyon_sayisi = 1 # Anlatımı sade tutmak için 1 iterasyon yeterli
ogrenme_orani = 0.1

for i in range(iterasyon_sayisi):
    print(f"\n########### ITERASYON {i+1} ###########\n")
    
    for veri in egitim_verisi:
        metin = veri["metin"]
        kaynak_idx = veri["kaynak_idx"]
        hedef_idx = veri["hedef_idx"]
        tokenler = metin.split()
        
        print(f"--- Girdi Metni: '{metin}' ---")
        print(f"Hedef: '{tokenler[kaynak_idx]}' kelimesi '{tokenler[hedef_idx]}' kelimesine dikkat etsin.\n")
        
        X = np.round(np.random.rand(len(tokenler), d_model), 2)
        print(f"Başlangıç Embedding Matrisi X (boyut: {X.shape}):\n{X}\n")

        # İleri Yayılım (Forward Pass)
        Q = np.dot(X, W_q)
        K = np.dot(X, W_k)
        V = np.dot(X, W_v)
        
        print(f"Oluşturulan Değer (V) Matrisi (Her kelimenin 'asıl içeriği'):\n{np.round(V,2)}\n")

        # Q ve K ile dikkat ağırlıklarını hesapla
        skorlar = np.dot(Q, K.T) / np.sqrt(d_k)
        dikkat_agirliklari = softmax(skorlar)
        
        print(f"--- Adım A: Dikkat Ağırlıklarının Hesaplanması (Q ve K kullanıldı) ---")
        print("Bu matris, 'nereye bakılacağını' ve 'ne kadar yoğunlukta bakılacağını' söyler.")
        print_attention(tokenler, dikkat_agirliklari)
        
        # --- YENİ EKLENEN ADIM: V MATRİSİNİN KULLANIMI ---
        print(f"--- Adım B: Final Çıktının Üretilmesi (V matrisi kullanıldı) ---")
        print("Şimdi, dikkat ağırlıkları (A) ile Değer matrisi (V) çarpılır: Çıktı = A @ V")
        print("Bu işlem, her kelime için diğer kelimelerin 'Değer'lerinin ağırlıklı bir ortalamasını alarak yeni bir vektör oluşturur.\n")
        
        final_cikti = np.dot(dikkat_agirliklari, V)

        print(f"ATTENTION KATMANININ GERÇEK FİNAL ÇIKTISI (boyut: {final_cikti.shape}):\n{np.round(final_cikti, 2)}\n")

        print("--- Çıktının Yorumu ---")
        kopek_orijinal_vektor = X[kaynak_idx]
        kopek_yeni_vektor = final_cikti[kaynak_idx]
        getirdi_deger_vektoru = V[hedef_idx]
        
        print(f"'{tokenler[kaynak_idx]}' kelimesinin orijinal vektörü: {kopek_orijinal_vektor}")
        print(f"'{tokenler[kaynak_idx]}' kelimesinin YENİ, bağlamsallaşmış vektörü: {np.round(kopek_yeni_vektor, 2)}")
        print("\nBu yeni vektör, artık sadece 'köpek' kelimesinin anlamını değil, aynı zamanda yüksek dikkat ettiği")
        print(f"'{tokenler[hedef_idx]}' kelimesinin Değer (V) vektörünün ({np.round(getirdi_deger_vektoru,2)}) anlamsal bilgisini de içinde barındırır.")
        print("İşte V matrisinin görevi, bu zenginleştirilmiş yeni temsilleri yaratmaktır!\n")
        print("-" * 50)

        # Hata ve Güncelleme Adımları (önceki gibi devam ediyor)
        mevcut_skor = dikkat_agirliklari[kaynak_idx, hedef_idx]
        hata = 1.0 - mevcut_skor
        duzeltme_miktari = hata * ogrenme_orani
        
        W_q += np.outer(X[kaynak_idx], Q[kaynak_idx]) * duzeltme_miktari
        W_k += np.outer(X[hedef_idx], K[hedef_idx]) * duzeltme_miktari
        
        toplam_dikkat = np.sum(dikkat_agirliklari, axis=0)
        en_onemli_kelime_idx = np.argmax(toplam_dikkat)
        W_v += np.outer(X[en_onemli_kelime_idx], V[en_onemli_kelime_idx]) * duzeltme_miktari
        print("Ağırlık matrisleri (W_q, W_k, W_v) güncellendi...\n")

print("\n\n--- EĞİTİM SONRASI FİNAL MATRİSLER ---")
print("Rastgele başlayan matrisler, hem 'etiketleri' (K) hem de 'içerikleri' (V) optimize edecek şekilde evrimleşti.")
print("Final W_q Matrisi:\n", np.round(W_q, 2))
print("\nFinal W_k Matrisi:\n", np.round(W_k, 2))
print("\nFinal W_v Matrisi:\n", np.round(W_v, 2))

--- BAŞLANGIÇ: RASTGELE MATRİSLER ---
Başlangıç W_q Matrisi:
 [[0.37 0.95 0.73]
 [0.6  0.16 0.16]
 [0.06 0.87 0.6 ]
 [0.71 0.02 0.97]]

Başlangıç W_k Matrisi:
 [[0.83 0.21 0.18]
 [0.18 0.3  0.52]
 [0.43 0.29 0.61]
 [0.14 0.29 0.37]]

Başlangıç W_v Matrisi:
 [[0.46 0.79 0.2 ]
 [0.51 0.59 0.05]
 [0.61 0.17 0.07]
 [0.95 0.97 0.81]] 


########### ITERASYON 1 ###########

--- Girdi Metni: 'akıllı köpek topu getirdi' ---
Hedef: 'köpek' kelimesi 'getirdi' kelimesine dikkat etsin.

Başlangıç Embedding Matrisi X (boyut: (4, 4)):
[[0.3  0.1  0.68 0.44]
 [0.12 0.5  0.03 0.91]
 [0.26 0.66 0.31 0.52]
 [0.55 0.18 0.97 0.78]]

Oluşturulan Değer (V) Matrisi (Her kelimenin 'asıl içeriği'):
[[1.02 0.84 0.46]
 [1.19 1.27 0.78]
 [1.14 1.15 0.52]
 [1.67 1.46 0.81]]

--- Adım A: Dikkat Ağırlıklarının Hesaplanması (Q ve K kullanıldı) ---
Bu matris, 'nereye bakılacağını' ve 'ne kadar yoğunlukta bakılacağını' söyler.
         akıllı    köpek     topu      getirdi 
---------------------------------------------

## 0.2. Basis ( Sorgu Örneği ) 

In [4]:
import numpy as np

# --- Fonksiyonlar ---
def softmax(x):
    e_x = np.exp(x - np.max(x)) # Numerik stabilite için
    return e_x / np.sum(e_x)

# --- Adım 1: "Eğitilmiş" Model ve Bilgi Bankası ---
print("--- Adım 1: 'Eğitilmiş' Modelin ve Bilgi Bankasının Hazırlanması ---")

# Önceki kodumuzdan elde ettiğimiz, artık rastgele olmayan, "eğitilmiş" ağırlık matrislerini kullanıyoruz.
W_q = np.array([[0.41, 0.44, 0.62], [0.5, 0.48, 0.51], [0.2, 0.21, 0.22], [0.21, 0.21, 0.23]])
W_k = np.array([[0.51, 0.46, 0.36], [0.65, 0.65, 0.52], [0.38, 0.32, 0.27], [0.4, 0.35, 0.3]])
W_v = np.array([[0.4, 0.65, 0.43], [0.49, 0.68, 0.49], [0.21, 0.24, 0.21], [0.22, 0.25, 0.23]])

print("Önceki eğitimden gelen 'bilge' W_q, W_k, W_v matrisleri yüklendi.\n")

# Modelin "bildiği" her şeyi içeren bilgi bankası (Context)
bilgi_bankasi_metni = "akıllı köpek topu getirdi yorgun kedi minderde uyudu"
context_tokenler = bilgi_bankasi_metni.split()

print(f"Bilgi Bankası (Context): '{bilgi_bankasi_metni}'")
print(f"Context Token'ları: {context_tokenler}\n")

# Bilgi bankasındaki her kelime için embedding oluşturalım (simülasyon)
d_model = 4
np.random.seed(10) # Farklı bir seed
X_context = np.random.rand(len(context_tokenler), d_model)

# BİLGİ BANKASININ K ve V MATRİSLERİNİ HESAPLA (Modelin Hafızası)
K_context = np.dot(X_context, W_k)
V_context = np.dot(X_context, W_v)

print("Bilgi bankası için K (Anahtar) ve V (Değer) matrisleri hesaplandı.")
print(f"K_context boyutu (hafızadaki etiketler): {K_context.shape}")
print(f"V_context boyutu (hafızadaki içerikler): {V_context.shape}\n")
print("="*60)


# --- Adım 2: Soruyu İşleme ---
print("\n--- Adım 2: Yeni Bir Soru Geldi! ---")
soru_metni = "kedi köpek ne yaptı"
soru_tokenler = soru_metni.split()
# Cevaplamak istediğimiz anahtar kelimeler
sorgu_kelimeleri = ["kedi", "köpek"] 

print(f"Soru: '{soru_metni}'")
print(f"Cevaplanacak anahtar kelimeler: {sorgu_kelimeleri}\n")

# Soru kelimeleri için embedding oluşturalım
X_soru = np.random.rand(len(soru_tokenler), d_model)

# SORU İÇİN Q MATRİSİNİ HESAPLA
Q_soru = np.dot(X_soru, W_q)
print("Soru için Q (Sorgu) matrisi hesaplandı.")
print(f"Q_soru boyutu: {Q_soru.shape}\n")
print("="*60)


# --- Adım 3: Cevap için Dikkat Mekanizmasını Çalıştırma ---
print("\n--- Adım 3: Dikkat Mekanizması ile Cevap Bulma ---")

for kelime in sorgu_kelimeleri:
    print(f"\n>>> SORGU: '{kelime}' kelimesi ne yaptı? <<<\n")
    
    # İlgili kelimenin Q vektörünü al
    kelime_index_in_soru = soru_tokenler.index(kelime)
    Q_tek_kelime = Q_soru[kelime_index_in_soru]
    
    print(f"1. '{kelime}' için SORGU (Q) vektörü alındı: {np.round(Q_tek_kelime, 2)}")
    
    # 2. Çapraz Dikkat Skorları Hesaplanıyor
    dikkat_skorlari = np.dot(Q_tek_kelime, K_context.T)
    
    print("\n2. Bu sorgu, bilgi bankasındaki her kelimeyle karşılaştırıldı (Sadece Q ve K kullanıldı).")
    
    # 3. Softmax ile Olasılık Dağılımı
    dikkat_agirliklari = softmax(dikkat_skorlari)
    print("\n3. Softmax ile 'dikkat ağırlıkları' bulundu (Nereye ne kadar dikkat etmeli?).")
    agirlik_raporu = "   "
    for i, token in enumerate(context_tokenler):
        agirlik_raporu += f"'{token}':{dikkat_agirliklari[i]:.3f}  "
    print(agirlik_raporu)

    # --- YENİ ve GÜNCEL ADIM 4: V MATRİSİ İLE CEVAP ÜRETME ---
    print("\n4. V Matrisi ile Cevap Üretimi (Attention'ın asıl çıktısı)")
    
    # Adım 4a: Bağlamsal Çıktı Vektörünü Sentezleme (A @ V)
    print("   4a. Dikkat ağırlıkları, V_context matrisiyle çarpılarak tek bir 'cevap' vektörü sentezlendi.")
    Cikti_vektoru = np.dot(dikkat_agirliklari, V_context)
    print(f"       -> Sentezlenen Çıktı Vektörü: {np.round(Cikti_vektoru, 2)}")

    # Adım 4b: En Benzer Kelimeyi Bulma
    print("\n   4b. Bu sentezlenmiş vektöre en çok benzeyen kelime hangisi?")
    # Bu yeni vektörün, bilgi bankasındaki hangi kelimenin 'Değer'ine (V) en çok benzediğini bulalım.
    # Benzerlik için dot product kullanabiliriz.
    benzerlik_skorlari = np.dot(V_context, Cikti_vektoru.T)
    cevap_index = np.argmax(benzerlik_skorlari)
    cevap_kelimesi = context_tokenler[cevap_index]
    print(f"       -> En Yüksek Benzerlik Skoruyla Bulunan Cevap: '{cevap_kelimesi}'")
    
    # --- YENİ ve GÜNCEL ADIM 5: SONUÇ ---
    print("\n5. Sonuç")
    print("-" * 20)
    print(f"SONUÇ: '{kelime}' ne yaptı? -> CEVAP: '{cevap_kelimesi}'")
    print("-" * 20)

print("\n\n--- ÖZET (GÜNCELLENDİ) ---")
print("Model, sorudaki kelimeleri birer SORG_U (Q) olarak kullandı.")
print("Bu sorguları, hafızasındaki ANAHTARLAR (K) ile karşılaştırarak dikkat ağırlıklarını belirledi.")
print("Bu ağırlıkları, hafızadaki asıl anlamsal içeriği taşıyan DEĞERLER (V) ile çarparak bağlamsal bir 'cevap vektörü' sentezledi.")
print("Son olarak, bu sentezlenmiş vektörü en iyi temsil eden kelimeyi bularak nihai cevabını verdi.")

--- Adım 1: 'Eğitilmiş' Modelin ve Bilgi Bankasının Hazırlanması ---
Önceki eğitimden gelen 'bilge' W_q, W_k, W_v matrisleri yüklendi.

Bilgi Bankası (Context): 'akıllı köpek topu getirdi yorgun kedi minderde uyudu'
Context Token'ları: ['akıllı', 'köpek', 'topu', 'getirdi', 'yorgun', 'kedi', 'minderde', 'uyudu']

Bilgi bankası için K (Anahtar) ve V (Değer) matrisleri hesaplandı.
K_context boyutu (hafızadaki etiketler): (8, 3)
V_context boyutu (hafızadaki içerikler): (8, 3)


--- Adım 2: Yeni Bir Soru Geldi! ---
Soru: 'kedi köpek ne yaptı'
Cevaplanacak anahtar kelimeler: ['kedi', 'köpek']

Soru için Q (Sorgu) matrisi hesaplandı.
Q_soru boyutu: (4, 3)


--- Adım 3: Dikkat Mekanizması ile Cevap Bulma ---

>>> SORGU: 'kedi' kelimesi ne yaptı? <<<

1. 'kedi' için SORGU (Q) vektörü alındı: [0.61 0.64 0.82]

2. Bu sorgu, bilgi bankasındaki her kelimeyle karşılaştırıldı (Sadece Q ve K kullanıldı).

3. Softmax ile 'dikkat ağırlıkları' bulundu (Nereye ne kadar dikkat etmeli?).
   'akıllı':0.117 